# Attention

_Deep Learning (CS230)_

**Let the model focus on the most relevant input parts, with weights that add to 1.**

When producing each output, a model shouldn't treat all inputs equally. Some words matter more right now.

     Attention lets the model focus. It gives each input part a weight, and the weights add up to 1.

---

This notebook is a practice scaffold for the **Attention** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# manual scaled dot-product attention
q = torch.randn(1, 4, 8)        # 4 queries, dim 8
k = torch.randn(1, 6, 8)        # 6 keys
v = torch.randn(1, 6, 8)        # 6 values
scores = q @ k.transpose(-2, -1) / (8 ** 0.5)   # similarity, scaled
weights = F.softmax(scores, dim=-1)             # rows sum to 1
out = weights @ v                                # weighted blend of values
print("attention weights sum to 1:", weights.sum(-1).round().tolist())
print("manual out:", tuple(out.shape))           # (1, 4, 8)

# the library version (multi-head)
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
attn_out, attn_w = mha(q, k, v)
print("multihead out:", tuple(attn_out.shape))

## Visualize the data & results

_Across the rows of a real digit image, which rows does self-attention make each row focus on?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
rows = digits.images[8].astype(float)    # 8 pixel-rows of a real 8

norm = np.linalg.norm(rows, axis=1)
scores = (rows @ rows.T) / (norm[:, None] * norm[None, :] + 1e-9) * 4   # query-key
e = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = e / e.sum(axis=1, keepdims=True)   # softmax per row -> attention

plt.imshow(weights, cmap="viridis")
plt.xticks(range(8), [f"row{i+1}" for i in range(8)], rotation=45)
plt.yticks(range(8), [f"row{i+1}" for i in range(8)])
plt.title("Self-attention over the 8 rows of a real digit image (each row sums to 1)")
plt.colorbar()
plt.show()